In [13]:
import os
import numpy as np
import cv2
from PIL import Image
import random
from tqdm import tqdm
from config import PATH_TEST, PATH_TRAINVAL, SEED, BATCH_SIZE

# ----------------------------
# CONFIGURATION
# ----------------------------
INPUT_IMAGE_DIR = "./path_to_dataset/color"
INPUT_MASK_DIR = "./path_to_dataset/label"



INPUT_IMAGE_DIR = os.path.join(PATH_TRAINVAL, 'color/')
INPUT_MASK_DIR = os.path.join(PATH_TRAINVAL, 'label/')


OUTPUT_DIR = "./point_prompt_dataset"
OUTPUT_IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
OUTPUT_MASK_DIR = os.path.join(OUTPUT_DIR, "masks")
OUTPUT_PROMPT_DIR = os.path.join(OUTPUT_DIR, "prompts")

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
os.makedirs(OUTPUT_PROMPT_DIR, exist_ok=True)

# ----------------------------
# UTILITY FUNCTIONS
# ----------------------------
def sample_point_from_mask(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    idx = np.random.randint(len(xs))
    return xs[idx], ys[idx]

def generate_heatmap(point, shape, sigma=5):
    heatmap = np.zeros(shape, dtype=np.float32)
    if point:
        x, y = point
        heatmap[y, x] = 1
        heatmap = cv2.GaussianBlur(heatmap, (0, 0), sigma)
        heatmap = heatmap / np.max(heatmap)
    return heatmap

# ----------------------------
# MAIN LOOP
# ----------------------------
image_list = sorted(os.listdir(INPUT_IMAGE_DIR))
mask_list = sorted(os.listdir(INPUT_MASK_DIR))

for img_name, mask_name in tqdm(zip(image_list, mask_list), total=len(image_list)):
    img_path = os.path.join(INPUT_IMAGE_DIR, img_name)
    mask_path = os.path.join(INPUT_MASK_DIR, mask_name)

    img = np.array(Image.open(img_path).convert("RGB"))
    mask = np.array(Image.open(mask_path))
    mask = (mask > 0).astype(np.uint8)  # Convert to binary if necessary

    point = sample_point_from_mask(mask)
    if point is None:
        continue

    heatmap = generate_heatmap(point, mask.shape)

    # Save outputs
    Image.fromarray(img).save(os.path.join(OUTPUT_IMAGE_DIR, img_name))
    Image.fromarray((mask * 255).astype(np.uint8)).save(os.path.join(OUTPUT_MASK_DIR, mask_name))
    np.save(os.path.join(OUTPUT_PROMPT_DIR, mask_name.replace(".png", ".npy")), heatmap)

print("✅ Preprocessing complete: saved images, masks, and prompts.")


100%|██████████| 3673/3673 [00:23<00:00, 153.27it/s]

✅ Preprocessing complete: saved images, masks, and prompts.
